In [5]:
import pandas as pd
import yaml
import importlib
utils = importlib.import_module("eval-app.src.utils")

class MyDumper(yaml.Dumper):

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, False)

In [6]:
def loadYML(file_path):
    with open(file_path) as stream:
        try:
            data = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
            data = {}
    return data

DIR_DATA = utils.Config.DIR_HOME / 'src' / 'evaluator' / 'data'

# Basic prompts

In [7]:
info = pd.read_csv(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / 'chemical_casrn.txt', delimiter=',', dtype='str')
tests = []
for i, row in info.iterrows():
    tests.append({'vars': {'chemical_and_CASRN': f'{row['Chemical']} (CAS number: {row['CASRN']})'}})

for species in ['human', 'rat']:

    with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / f'basic-prompts_{species}_tests.yaml', 'w') as outfile:
        yaml.dump({'tests': tests}, outfile)
    
    with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / f'basic-prompts_{species}.txt') as f:
        prompts = [x.strip() for x in f.readlines()]

    with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / f'basic-prompts_{species}.yaml', 'w') as outfile:
        yaml.dump({'prompts': prompts}, outfile)

# Toxicity type prompts

### Check similarity using a python script

In [8]:
import copy

test_template_no_assertion = {
    'description': 'Test similarity',
    'vars': {
        'chemical_and_CASRN': 'Aspirin (CAS number: 50-78-2)',
        'effect_type': 'Sub acute toxicity'
    }
}

test_template_assertion = {
    'description': 'Test similarity',
    'vars': {
        'chemical_and_CASRN': 'Aspirin (CAS number: 50-78-2)',
        'effect_type': 'Sub acute toxicity'
    },
    'assert': [
        {
            'expected_phrases': ''
        }
    ]
}

def createTests(info, species):

    tests = []

    for i, row in info.iterrows():
        for col in info.columns:
            if col in ['Other', 'Toxicity']: continue
            test_temp = copy.deepcopy(test_template_assertion if row[col] else test_template_no_assertion)
            chemname, casrn = col.split(', ')
            test_temp['vars']['chemical_and_CASRN'] = f'{chemname} (CAS number: {casrn})'
            tox_type = row['Toxicity'].strip()
            if tox_type:
                test_temp['vars']['effect_type'] = f"{tox_type} toxicity"
            else:
                test_temp['vars']['effect_type'] = row['Other'].strip()
            if row[col]:
                test_temp['assert'][0]['expected_phrases'] = str(row[col]).split('; ')
            tests.append(test_temp)

    with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / f'tox-type-assertion-prompts_{species}.yaml', 'w') as outfile:
        prompts = ["Answer the following question with a list of string. List the {effect_type} of {chemical_and_CASRN} on " + species]
        yaml.dump({'prompts': prompts}, outfile, Dumper=MyDumper, default_flow_style=False)

    with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / f'tox-type-assertion-prompts_{species}_tests.yaml', 'w') as outfile:
        yaml.dump({'tests': tests}, outfile, Dumper=MyDumper, default_flow_style=False)


### Human

In [9]:
info = pd.read_csv(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / 'tox-type-prompts_human.txt', delimiter='\t', dtype='str')
info[info.isna()] = ''
info.head()

,Other,Toxicity,"Aspirin, 50-78-2","Paracetamol, 103-90-2","Metformin, 657-24-9","Atorvastatin, 134523-00-5","Prednisone, 53-03-2","Dioxin, 1746-01-6","Benzene, 71-43-2","Arsenite, 15502-74-6",...,"6PPQ (N-(1,3-dimethylbutyl)-N'-phenyl-p-phenylenediamine), 2754428-18-5","Phenanthrene, 85-01-8","TPHP (triphenylphosphate), 115-86-6","Lanthanum, 7439-91-0","Galaxolide, 1222-05-5","1,6-Dimethyl-3,4-dihydroisoquinoline, 91753-09-2","2-Methyl-4-(4-methylphenyl)-2,3-dihydro-1,5-benzothiazepine, 74148-62-2","N-Cyclopropylmethyl-10,11-dihydro-5H-dibenzo-(a,d)-cyclohepten-5-imine, 59864-46-9","4-Methyl-1,2-dihydrobenzo[f]isoquinoline, 29248-42-8","6-Methyl-2,5-diphenyl-6H-1,3,4-thiadiazine, 62625-70-1"
0,,Sub acute,Gastrointestinal bleeding,Hypothermia; Acidosis; Hepatocellular/liver in...,Acidosis,,Weight gain; Increased appetite,Altered liver function,,Fatty degeneration of the heart,...,,,,,,,,,,
1,,Chronic,Gastrointestinal bleeding; Anemia; Chronic kid...,,,,Osteoporosis; Weight gain; Increased appetite;...,Breast cancer; Endometrial cancer; Testis cancer,Anemia; Bone marrow suppression; Immune suppre...,Anemia; Leukopenia; Liver damage; Hyperpigment...,...,,,,,,,,,,
2,,Developmental,Gastroschisis,,,,,,,,...,,,,,,,,,,
3,,Immune/lymphatic system,Hypersensitivity/allergic reaction,,,,Edema; Immune suppression,Immune suppression,,,...,,,,,,,,,,
4,,Reproductive system,,,,,,Endometriosis; Decreased sperm count,,,...,,,,,,,,,,


In [12]:
createTests(info, species='human')

### Rat

In [10]:
info = pd.read_csv(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / 'tox-type-prompts_rat.txt', delimiter='\t', dtype='str')
info[info.isna()] = ''
info.head()

,Other,Toxicity,"Aspirin, 50-78-2","Paracetamol, 103-90-2","Metformin, 657-24-9","Atorvastatin, 134523-00-5","Prednisone, 53-03-2","Dioxin, 1746-01-6","Benzene, 71-43-2","Arsenite, 15502-74-6",...,"6PPQ (N-(1,3-dimethylbutyl)-N'-phenyl-p-phenylenediamine), 2754428-18-5","Phenanthrene, 85-01-8","TPHP (triphenylphosphate), 115-86-6","Lanthanum, 7439-91-0","Galaxolide, 1222-05-5","1,6-Dimethyl-3,4-dihydroisoquinoline, 91753-09-2","2-Methyl-4-(4-methylphenyl)-2,3-dihydro-1,5-benzothiazepine, 74148-62-2","N-Cyclopropylmethyl-10,11-dihydro-5H-dibenzo-(a,d)-cyclohepten-5-imine, 59864-46-9","4-Methyl-1,2-dihydrobenzo[f]isoquinoline, 29248-42-8","6-Methyl-2,5-diphenyl-6H-1,3,4-thiadiazine, 62625-70-1"
0,,Sub acute,Gastrointestinal bleeding,,,,,,,,...,,,,,,,,,,
1,,Subchronic,,"Liver inflammation; Hepataytomegaly; Kidney, T...",parotid salivary gland necrosis; metabolic ac...,,Adrenal atrophy; Lypmphatic organ atrophy;,,decreased femoral bone marrow cellularity;,Bone marrow suppresion; Liver inflammatory cel...,...,,,,,,,,,,
2,,Chronic,Angiosarcoma of the liver,Follicular cell hyperplasia,,,,"Thyroid gland, Follicular Cell Hypertrophy; He...",Palate: Squamous Cell Papilloma; Spleen Lympho...,,...,,,,,,,,,,
3,,Developmental,,,,,,,,,...,,,,,,,,,,
4,,Immune/lymphatic system,,,,,,,,,...,,,,,,,,,,


In [11]:
createTests(info, species='rat')

# ABT QA

In [13]:
info = pd.read_json(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / 'abt_qa_wo_noise.json')
info[info.isna()] = ''
info

,Question Index,Questions,Options,Question Source,Answers,Answer Source
0,2000A Q1,Which of the following pairs of radionuclides ...,"{'A': '222 Rn and 137 Cs', 'B': '222 Rn and 90...",ABT2000#17.pdf,[[C]],"ABT #17.pdf, Document.pdf"
1,2000A Q2,A semi-conscious pesticide sprayer is brought ...,{'A': 'propanolol followed by an intravenous a...,ABT2000#17.pdf,[[D]],"ABT #17.pdf, Document.pdf"
2,2000A Q3,Botulinum toxin inhibits neuromuscular communi...,{'A': 'blocking sodium channels along the moto...,ABT2000#17.pdf,[[B]],"ABT #17.pdf, Document.pdf"
3,2000A Q5,Contact urticaria is characteristic of,"{'A': 'mistletoe', 'B': 'crocus', 'C': 'pokewe...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf",[[D]],"ABT #17.pdf, Document.pdf"
4,2000A Q6,The dominant route of excretion for absorbed l...,"{'A': 'bile', 'B': 'feces', 'C': 'urine', 'D':...","ABT #17.pdf, Document.pdf, ABT2000#17.pdf",[[C]],"ABT #17.pdf, Document.pdf"
...,...,...,...,...,...,...
469,2007C Q36,"The following factors, identified by Bradford-...","{'A': 'consistency of the association', 'B': '...",24-2007-recert no answers.pdf,[[E]],ABT Recert 24-2007 MH group Answers.doc
470,2007C Q37,The probability that a chemical may produce ca...,{'A': 'whether comparable peak plasma concentr...,24-2007-recert no answers.pdf,"[[A, E]]",ABT Recert 24-2007 MH group Answers.doc
471,2007C Q38,Elevation of which of the following serum/plas...,"{'A': 'troponin I', 'B': 'alkaline phosphatase...",24-2007-recert no answers.pdf,[[C]],ABT Recert 24-2007 MH group Answers.doc
472,2007C Q39,Short Term Exposure Limits (STELs) are,{'A': 'the absolute maximum concentrations to ...,24-2007-recert no answers.pdf,[[C]],ABT Recert 24-2007 MH group Answers.doc


In [14]:
import copy

test_template = {
    'description': 'Test similarity',
    'vars': {
        'question_with_options': ''
    },
    'assert': [
        {
            'expected_phrases': ''
        }
    ]
}

tests = []

answers_noisy = []
for i, row in info.iterrows():
    if len(row['Answers']) > 1: 
        continue
    try:
        test_temp = copy.deepcopy(test_template)
        options = '\n\n'.join([f'({k}) {v}' for k, v in row['Options'].items()])
        test_temp['vars']['question_with_options'] = f'**Question**:\n{row['Questions']}\n**Options**:\n{options}'
        test_temp['assert'][0]['expected_phrases'] = list(map(lambda x: f'({x.strip()}) {row['Options'][x.strip()]}', row['Answers'][0]))
    except Exception as exp:
        answers_noisy.append(row)
        continue
    tests.append(test_temp)

with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / f'abt-qa-assertion-prompts_mixed.yaml', 'w') as outfile:
    prompts = ['Consider the following question followed by options. Answer the question from choosing the options. You can choose multiple options. Each option must start with the option number.\n{question_with_options}']
    yaml.dump({'prompts': prompts}, outfile, Dumper=MyDumper, default_flow_style=False)

with open(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'abt-qa-assertion-prompts_mixed_tests.yaml', 'w') as outfile:
    yaml.dump({'tests': tests}, outfile, Dumper=MyDumper, default_flow_style=False)

pd.DataFrame(answers_noisy)

""


# Reset eval configs

In [21]:
import yaml
import tqdm
import importlib
mdb = importlib.import_module("eval-app.src.evaluator.src.evaluation.db")

def loadYML(file_path):
    data = None
    with open(file_path) as stream:
        try:
            data = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)
    return data

config_path = DIR_DATA / 'toxpipe_eval_info' / 'config'

system_prompt = loadYML(config_path / 'raw' / 'system_prompt.yaml')
providers = loadYML(config_path / 'raw' / 'providers.yaml')

for prompt_type in ['basic-prompts', 'tox-type-assertion-prompts']:
    for species in ['human', 'rat']:
        print(f'Processing {prompt_type}_{species}')
        tests = loadYML(config_path / f'{prompt_type}_{species}_tests.yaml')['tests']
        prompts_vars_asserts = []
        for prompt in loadYML(config_path / f'{prompt_type}_{species}.yaml')['prompts']:
            prompts_vars_asserts.append({'prompt': prompt, 'tests': tests})

        for framework_type in tqdm.tqdm(['base-model', 'rag', 'agentic']):
            d = {'description': f'Tests on {framework_type} framework with {prompt_type} prompts for {species}'}
            d |= system_prompt
            d |= providers[framework_type]
            d['prompts_vars_asserts'] = prompts_vars_asserts.copy()

            eval_name = f'{framework_type}_{prompt_type}_{species}'

            db = mdb.EvalConfigDB(eval_name)
            db.drop()
            db.add(d)
        
for prompt_type in ['abt-qa-assertion-prompts']:
    print(f'Processing {prompt_type}_mixed')
    tests = loadYML(config_path / f'{prompt_type}_mixed_tests.yaml')['tests']
    prompts_vars_asserts = []
    for prompt in loadYML(config_path / f'{prompt_type}_mixed.yaml')['prompts']:
        prompts_vars_asserts.append({'prompt': prompt, 'tests': tests})
    for framework_type in tqdm.tqdm(['base-model', 'rag', 'agentic']):
        d = {'description': f'Tests on {framework_type} framework with {prompt_type} prompts'}
        d |= system_prompt
        d |= providers[framework_type]
        d['prompts_vars_asserts'] = prompts_vars_asserts.copy()

        eval_name = f'{framework_type}_{prompt_type}_mixed'

        db = mdb.EvalConfigDB(eval_name)
        db.drop()
        db.add(d)
        
        #with open(utils.Config.DIR_HOME / 'tests' / f'{framework_type}_{prompt_type}_mixed' / 'config.yaml', 'w') as outfile:
        #    yaml.dump(d, outfile, Dumper=MyDumper, default_flow_style=False)        

Processing basic-prompts_human


100%|██████████| 3/3 [00:01<00:00,  2.49it/s]


Processing basic-prompts_rat


100%|██████████| 3/3 [00:01<00:00,  2.45it/s]


Processing tox-type-assertion-prompts_human


100%|██████████| 3/3 [00:01<00:00,  2.43it/s]


Processing tox-type-assertion-prompts_rat


100%|██████████| 3/3 [00:01<00:00,  2.44it/s]


Processing abt-qa-assertion-prompts_mixed


100%|██████████| 3/3 [00:01<00:00,  2.00it/s]


## Fix evals and configs in mongodb

In [ ]:
import tqdm
import importlib
mdb = importlib.import_module("eval-app.src.evaluator.src.evaluation.db")

for framework_type in tqdm.tqdm(['base-model', 'rag', 'agentic']):
    for species in ['human', 'rat']:
        eval_name = f'{framework_type}_tox-type-assertion-prompts_{species}'
        db = mdb.EvalDB(eval_name)
        prompts = db.collection.distinct('prompt')
        for p in prompts:
            db.collection.update_many(
                    {'prompt': p},  # Empty filter to select all documents
                    {'$set': {'prompt': p.replace('chem_casrn', 'chemical_and_CASRN')}}
                )

100%|██████████| 3/3 [00:01<00:00,  1.58it/s]


In [ ]:
db = mdb.EvalDB('base-model_tox-type-assertion-prompts_rat')
db.collection.distinct('prompt')

['Answer the following question with a list of string. List the {effect_type} of {chemical_and_CASRN} on human']

In [7]:
info = pd.read_csv(DIR_DATA / 'toxpipe_eval_info' / 'config' / 'raw' / 'chemical_casrn.txt', delimiter=',', dtype='str')

for llm in tqdm.tqdm(['azure-o3', 'azure-gpt-4o', 'claude-3-7-sonnet', 'gemini-2.5-pro', 'gemini-2.5-flash', 'llama3-3-70b', 'mistral-large-2']):
    for framework_type in tqdm.tqdm(['base-model']):#, 'rag', 'agentic']):
        for eval_set in ['basic-prompts_human', 'basic-prompts_rat', 
                            'tox-type-assertion-prompts_human', 'tox-type-assertion-prompts_rat',
                            'abt-qa-assertion-prompts_mixed']:
            eval_name = f'{framework_type}_{eval_set}'
            db = mdb.EvalDB(eval_name)
            for i, row in info.iterrows():
                db.collection.update_many(
                        {'provider.id': f'openai:chat:{llm}'},
                        {'$set': {'provider.id': f'llm:openai:chat:{llm}'}}
                )

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:43<00:00,  6.15s/it]
